In [1]:
!pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

#**Import Drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#**Google API Authentication**

In [4]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
SCOPES = ["https://www.googleapis.com/auth/drive",
          "https://www.googleapis.com/auth/forms.body"]

#Connecting to Google API Key file
SERVICE_ACCOUNT_FILE = "/content/drive/My Drive/TADATASET/form-generator-project-ac479f21d09f.json"
creds = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
drive_service = build('drive', 'v3', credentials=creds)
form_service = build('forms', 'v1', credentials=creds)

#**Process**

In [5]:
import os

#Create pre-testing dataset folders
base_directory = '/content/drive/My Drive/TADATASET/Sewudino/AudioData'
total_data = 793

for i in range(0, total_data, 100):
    start_range = i + 1
    end_range = min(i + 100, total_data)
    folder_name = f"{start_range}-{end_range}"
    folder_path = os.path.join(base_directory, folder_name)
    os.makedirs(folder_path, exist_ok=True)
    print(f"Created folder: {folder_path}")

Created folder: /content/drive/My Drive/TADATASET/Sewudino/AudioData/1-100
Created folder: /content/drive/My Drive/TADATASET/Sewudino/AudioData/101-200
Created folder: /content/drive/My Drive/TADATASET/Sewudino/AudioData/201-300
Created folder: /content/drive/My Drive/TADATASET/Sewudino/AudioData/301-400
Created folder: /content/drive/My Drive/TADATASET/Sewudino/AudioData/401-500
Created folder: /content/drive/My Drive/TADATASET/Sewudino/AudioData/501-600
Created folder: /content/drive/My Drive/TADATASET/Sewudino/AudioData/601-700
Created folder: /content/drive/My Drive/TADATASET/Sewudino/AudioData/701-793


In [7]:
def get_subfolder_links(service, parent_folder_id):
    query = f"'{parent_folder_id}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    results = service.files().list(
        q=query,
        fields="files(id, name)",
        pageSize=1000
    ).execute()
    subfolders = results.get('files', [])
    links = []
    for folder in subfolders:
        folder_id = folder['id']
        folder_name = folder['name']
        folder_link = f"{folder_id}"
        links.append((folder_name, folder_link))
    return links
parent_id = "1Z2LtUVg-Fuh1uJ83hZbae5HPNxQK9dKs"
links = get_subfolder_links(drive_service, parent_id)
for name, link in links:
    print(f"{name}: {link}")

701-793: 1LBUzvUsxJGubkP8Zm8Sid_vQeUI-i1Qh
601-700: 1Syr88fCPMFPOyIDM3cB9SZUVfipDm3sw
501-600: 1CrkG3szjWBnOnEwfYsBWM5IkgTuiKaij
401-500: 1_zQC7vWb89X9G2va2gy_EamTvGQWYJnZ
301-400: 1xED6jhr-iIODX9lwLoJYd-LMiIa9Niew
201-300: 1IXgCUWEM8dg1UjRphq1xFqabB5BIGnXU
101-200: 1Q0OHB6ofKgBqDst_I3vdlvVSBjl6b3rC
1-100: 1dy77N1at_AWI3dPe-53DYPBtwoiui_09


In [8]:
import re

def distribusikan_file(service):
    # 1. Senarai ID Folder mengikut julat
    folder_mapping = {
        "701-793": "1LBUzvUsxJGubkP8Zm8Sid_vQeUI-i1Qh",
        "601-700": "1Syr88fCPMFPOyIDM3cB9SZUVfipDm3sw",
        "501-600": "1CrkG3szjWBnOnEwfYsBWM5IkgTuiKaij",
        "401-500": "1_zQC7vWb89X9G2va2gy_EamTvGQWYJnZ",
        "301-400": "1xED6jhr-iIODX9lwLoJYd-LMiIa9Niew",
        "201-300": "1IXgCUWEM8dg1UjRphq1xFqabB5BIGnXU",
        "101-200": "1Q0OHB6ofKgBqDst_I3vdlvVSBjl6b3rC",
        "1-100": "1dy77N1at_AWI3dPe-53DYPBtwoiui_09"
    }

    # ID folder "baru" (sumber fail)
    baru_folder_id = "1qzqfmmrNeF3pB7eV-Z5KACLxHB89Eodg"

    print("Membaca fail dari folder 'baru'...")

    # 2. Ambil semua fail dari folder "baru"
    query = f"'{baru_folder_id}' in parents and mimeType != 'application/vnd.google-apps.folder' and trashed = false"
    files = []
    page_token = None

    while True:
        results = service.files().list(
            q=query,
            fields="nextPageToken, files(id, name)",
            pageSize=1000,
            pageToken=page_token
        ).execute()

        files.extend(results.get('files', []))
        page_token = results.get('nextPageToken', None)
        if not page_token:
            break

    print(f"Ditemui jumlah {len(files)} fail. Memulakan proses MOVE (Pindah)...\n")

    # 3. Proses pemindahan berdasarkan nombor terakhir pada nama fail
    for file in files:
        file_id = file['id']
        file_name = file['name']

        # Cari SEMUA kumpulan nombor dalam nama fail
        numbers = re.findall(r'\d+', file_name)

        if not numbers:
            print(f"⚠️ Abaikan '{file_name}': Tiada nombor dikesan.")
            continue

        # Ambil nombor yang PALING TERAKHIR (contoh: dari "01" dan "0982", ia ambil "0982")
        file_num = int(numbers[-1])

        target_folder_id = None
        target_folder_name = ""

        # Padankan nombor dengan julat folder
        for f_name, f_id in folder_mapping.items():
            start_num, end_num = map(int, f_name.split('-'))
            if start_num <= file_num <= end_num:
                target_folder_id = f_id
                target_folder_name = f_name
                break

        # 4. Eksekusi Pindah (Move)
        if target_folder_id:
            try:
                service.files().update(
                    fileId=file_id,
                    addParents=target_folder_id,
                    removeParents=baru_folder_id,
                    fields='id, parents'
                ).execute()
                print(f"✅ Dipindah: '{file_name}' --> Folder '{target_folder_name}'")
            except Exception as e:
                 print(f"❌ Gagal memindah '{file_name}': {e}")
        else:
            # Jika ada fail seperti 984 atau ke atas, ia tidak akan dipindahkan
            print(f"❌ Gagal: Nombor {file_num} pada fail '{file_name}' berada di luar julat (1-983).")

    print("\n🎉 Proses Pindah Selesai!")

# ==========================================
# JALANKAN PROGRAM
# (Pastikan 'drive_service' sudah disahkan)
# ==========================================
distribusikan_file(drive_service)

Membaca fail dari folder 'baru'...
Ditemui jumlah 793 fail. Memulakan proses MOVE (Pindah)...

✅ Dipindah: '01-SWDN-UT-0793.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0792.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0791.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0790.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0789.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0788.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0787.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0786.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0785.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0784.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0783.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0782.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0781.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0780.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0779.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0778.wav' --> Folder '701-793'
✅ Dipindah: '01-SWDN-UT-0

In [ ]:
import re

#Reading audio files in the Drive folder
def drivefilelinks(folder_id):
    query = f"'{folder_id}' in parents and mimeType != 'application/vnd.google-apps.folder'"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get("files", [])
    file_links = []
    base_url = "https://drive.google.com/file/d/"
    for file in sorted(files, key=lambda x: x['name']):
        file_id = file["id"]
        file_name = file["name"]
        file_link = f"{base_url}{file_id}/view"
        file_links.append((file_name, file_link))
    return file_links

#Reading audio file links in the folder
film = "Pabrik Gula"
film_code = "PKGL"
folder_id = "1i-N_dV6_wKsnDqU0KsXOPOKRD3E-G0D2"

# Get the folder name from the folder_id
folder_metadata = drive_service.files().get(fileId=folder_id, fields='name').execute()
folder_name = folder_metadata.get('name')

# Extract first and last numbers from the folder name
first_number = 0
last_number = 0
if folder_name:
    match = re.match(r'(\d+)-(\d+)', folder_name)
    if match:
        first_number = int(match.group(1))
        last_number = int(match.group(2))
        print(f"Folder name : {folder_name}")
    else:
        print(f"Warning: Could not extract numbers from folder name: {folder_name}")

file_links = drivefilelinks(folder_id)
print(f"total data : {len(file_links)}")
for name, link in file_links:
    print(name, ":", link)